In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

import keras
import keras_cv

print("GPUs:", gpus)

2026-04-30 21:52:42.823955: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-30 21:52:42.838268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777578762.855188 2053778 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777578762.860081 2053778 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-30 21:52:42.879592: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
class CFG:
    verbose = 2  # Verbosity
    seed = 48  # Random seed
    neg_sample = 0.01 # Downsample negative class
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [256, 256]  # Input image size
    epochs = 4 # Training epochs
    batch_size = 128  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [3]:
BASE_PATH = "/home/cgiovannoni/projects/isic24-dum"
DATA_PATH = f"{BASE_PATH}/data/new"
PROC_PATH = f"{BASE_PATH}/models/enhanced/enhanced/preprocessed_dataset"
ART_PATH = f"{BASE_PATH}/artifacts/enhanced"
FEATURE_SPACE_PATH = f"{ART_PATH}/feature_space"
RUNS_PATH = f"{ART_PATH}/runs"
SUMMARIES_PATH = f"{ART_PATH}/summaries"

In [4]:
import pandas as pd

# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [5]:
print(df['target'].value_counts())

target
0    363119
1       364
Name: count, dtype: int64


In [5]:
print("Class Distribution Before Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

# Sampling
majority_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
minority_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([majority_df, minority_df], axis=0).sample(frac=1.0)

print("\nClass Distribution After Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

Class Distribution Before Sampling (%):
target
0    99.899858
1     0.100142
Name: proportion, dtype: float64

Class Distribution After Sampling (%):
target
0    66.611631
1    33.388369
Name: proportion, dtype: float64


In [6]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.7506196640044065), 1: np.float64(1.4975274725274725)}


In [7]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [8]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

# Num Train: 4356 | Num Valid: 1095


In [9]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

training:
target
0    2904
1    1452
Name: count, dtype: int64

validation:
target
0    727
1    368
Name: count, dtype: int64


# 📊 | Tabular Features

### Enhanced


In [10]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [11]:
def build_augmenter():
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)
    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)

    # Decode before cache (so we don't decode every epoch)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed, reshuffle_each_iteration=True)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        opt.experimental_slack = True
        ds = ds.with_options(opt)

    ds = ds.batch(batch_size, drop_remainder=drop_remainder)

    # Keep augmentation AFTER cache so it's still random per epoch
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTO)

    ds = ds.prefetch(AUTO)
    return ds

def _to_feature_dict(df):
    feat_dict = {}
    for c in df.columns:
        s = df[c]
        if s.dtype == object or str(s.dtype).startswith('string'):
            feat_dict[c] = s.astype(str).to_numpy()
        else:
            feat_dict[c] = s.to_numpy()
    return feat_dict

In [12]:
from tqdm import tqdm

# Build datasets
print("# Training:")
training_features = _to_feature_dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(
    training_ids, training_validation_hdf5, training_features, training_labels,
    batch_size=CFG.batch_size, shuffle=True, augment=True
)

print("# Validation:")
validation_features = _to_feature_dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(
    validation_ids, training_validation_hdf5, validation_features, validation_labels,
    batch_size=CFG.batch_size, shuffle=False, augment=False
)

print("# Testing:")
testing_features = _to_feature_dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(
    testing_ids, testing_hdf5, testing_features,
    batch_size=CFG.batch_size, shuffle=False, augment=False, cache=False
)

# Training:


I0000 00:00:1777578789.891415 2053778 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22560 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
Loading Images : 100%|██████████| 4356/4356 [00:00<00:00, 5399.03it/s]


# Validation:


Loading Images : 100%|██████████| 1095/1095 [00:00<00:00, 6025.47it/s]


# Testing:


Loading Images : 100%|██████████| 37576/37576 [00:06<00:00, 5885.59it/s]


In [14]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

W0000 00:00:1777578853.614674 2053778 slack.cc:103] Could not find a final `prefetch` in the input pipeline to which to introduce slack.
2026-04-30 21:54:15.353467: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: CANCELLED: RecvAsync is cancelled.
	 [[{{node random_apply_1_2/map/while/body/_37/random_apply_1_2/map/while/cond/pivot_t/_139/_65}}]] [type.googleapis.com/tensorflow.DerivedStatus='']
2026-04-30 21:54:15.353529: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: CANCELLED: RecvAsync is cancelled.
	 [[{{node random_apply_1_2/map/while/body/_37/random_apply_1_2/map/while/cond/pivot_t/_139/_65}}]]
	 [[random_apply_1_2/map/while/body/_37/random_apply_1_2/map/while/cond/then/_135/random_apply_1_2/map/while/cond/random_flip_1/stateful_uniform/RngReadAndSkip/_70]] [type.googleapis.com/tensorflow.DerivedStatus='']
2026-04-30 21:54:15.353543: I tensorflow/core/framework/local_rendezvous.cc:424] 

(None, 130)


In [15]:
os.makedirs(FEATURE_SPACE_PATH, exist_ok=True)
filepath = os.path.join(FEATURE_SPACE_PATH, "feature_space.keras")

feature_space.save(filepath)
print(f"FeatureSpace saved at: '{filepath}'")

FeatureSpace saved at: '/home/cgiovannoni/projects/isic24-dum/artifacts/enhanced/feature_space/feature_space.keras'


In [16]:
# Apply Feature Processing
augment_fn = build_augmenter()

auto = tf.data.AUTOTUNE

training_ds = training_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
training_ds = training_ds.map(augment_fn, num_parallel_calls=auto)
training_ds = training_ds.prefetch(auto)

validation_ds = validation_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
validation_ds = validation_ds.prefetch(auto)

testing_ds = testing_ds.map(
    lambda x: {'images': x['images'], 'features': feature_space(x['features'])},
    num_parallel_calls=auto,
)
testing_ds = testing_ds.prefetch(auto)

In [17]:
for x, y in training_ds.take(1):
    print(type(x))
    print(type(x["features"]))
    if isinstance(x["features"], dict):
        print({k: v.shape for k, v in x["features"].items()})
    else:
        print(x["features"].shape, x["features"].dtype)

<class 'dict'>
<class 'tensorflow.python.framework.ops.EagerTensor'>
(128, 130) <dtype: 'float32'>


In [18]:
# optimizer, loss & metric
adam = keras.optimizers.Adam(learning_rate=1e-4)
auc = keras.metrics.AUC(name='auc')
bce = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

## Modelling

### callbacks

In [19]:
import math
import matplotlib.pyplot as plt

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

#### shap

In [20]:

_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])


def aggregate_raw_importance(encoded_feature_names, mean_abs_vec):
    agg = {}
    for j, val in enumerate(mean_abs_vec):
        name = encoded_feature_names[j]
        if "__" in name:
            raw = name.split("__", 1)[0]
        elif name.startswith("age_approx_bin_"):
            raw = "age_approx"
        else:
            raw = name
        agg.setdefault(raw, 0.0)
        agg[raw] += float(abs(val))
    return agg



def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    # ax.bar_label(container, fmt="{:.4f}", padding=4, fontsize=7)
    # adaptive barplot decimal precision
    for container in ax.containers:
        labels = []
        for v in container.datavalues:
            if abs(v) < 1e-3:
                labels.append(f"{v:.2e}")
            else:
                labels.append(f"{v:.4f}")
        ax.bar_label(container, labels=labels, padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]

def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
    ):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


import shap
from typing import Tuple, Dict, Any

class ShapEngine:
    """
    SHAP engine for the tabular 'features' input of the multimodal model.

    - Original model signature: model({"images": images, "features": features}) -> (batch, 1)
    - Internal wrapper: wrapped_model([features, images]) -> (batch, 1)
    - Explainer: shap.GradientExplainer on wrapped_model
    - We keep SHAP values only for the 'features' input (index 0 in [features, images]).
    """

    def __init__(
        self,
        background_size: int = 128,
        max_explain_samples: int = 64,
        top_k: int = 10,
        positive_threshold: float = 0.5,
        use_gradient_explainer: bool = True,
    ):
        self.background_size = background_size
        self.max_explain_samples = max_explain_samples
        self.top_k = top_k
        self.positive_threshold = positive_threshold
        self.use_gradient_explainer = use_gradient_explainer

        self._explainer = None
        self._wrapped_model = None   # Keras model: [features, images] -> original_model(...)
        self._feature_input_index = 0

    # ----------------- internal utils ----------------- #
    def _build_wrapped_model(
        self,
        base_model: tf.keras.Model,
        feature_shape,
        image_shape,
    ):
        """
        Build a Keras model that takes [features, images] and calls the original
        dict-based model internally.
        """
        feat_in = tf.keras.Input(shape=feature_shape[1:], name="shap_features")
        img_in = tf.keras.Input(shape=image_shape[1:], name="shap_images")

        out = base_model({"features": feat_in, "images": img_in})
        self._wrapped_model = tf.keras.Model(
            inputs=[feat_in, img_in],
            outputs=out,
            name="wrapped_for_shap",
        )

    def _build_explainer(
        self,
        base_model: tf.keras.Model,
        bg_features: np.ndarray,
        bg_images: np.ndarray,
    ):
        # Build wrapped model lazily
        if self._wrapped_model is None:
            self._build_wrapped_model(
                base_model,
                feature_shape=bg_features.shape,
                image_shape=bg_images.shape,
            )

        bg_inputs = [bg_features, bg_images]  # [features, images]

        if self.use_gradient_explainer:
            self._explainer = shap.GradientExplainer(self._wrapped_model, bg_inputs)
        else:
            self._explainer = shap.explainers.Deep(self._wrapped_model, bg_inputs)

    def _select_background_and_expl_set(
        self,
        features: np.ndarray,
        images: np.ndarray,
    ):
        n = features.shape[0]
        bg_idx = np.random.choice(n, size=min(self.background_size, n), replace=False)

        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.random.choice(n, size=self.max_explain_samples, replace=False)

        bg_features = features[bg_idx]
        bg_images = images[bg_idx]
        ex_features = features[ex_idx]
        ex_images = images[ex_idx]

        return (bg_features, bg_images), (ex_features, ex_images), ex_idx

    # ----------------- public API ----------------- #
    def explain(
        self,
        validation_data: Tuple[np.ndarray, np.ndarray, np.ndarray],
        model: tf.keras.Model,
        class_index: int = 0,  # irrelevant for scalar output
    ) -> Tuple[Dict[str, Any], np.ndarray, np.ndarray]:
        """
        validation_data: (features, images, labels) (all numpy arrays)

        Returns:
            stats: dict with
                - mean_abs_all: (n_features,)
                - mean_abs_pos: (n_features,)
                - mean_abs_neg: (n_features,)
                - topk_all_indices, topk_pos_indices, topk_neg_indices
                - explained_indices: indices into the original batch
            shap_values_features: (n_explain, n_features)
            preds: (n_explain, 1)
        """

        features, images, labels = validation_data

        (bg_features, bg_images), (ex_features, ex_images), ex_idx = \
            self._select_background_and_expl_set(features, images)

        # Lazy explainer init on wrapped model
        if self._explainer is None:
            self._build_explainer(model, bg_features, bg_images)

        ex_inputs = [ex_features, ex_images]  # [features, images]
        preds = self._wrapped_model(ex_inputs).numpy()  # (n_ex, 1)

        # SHAP values: list [phi_features, phi_images]
        shap_values_all_inputs = self._explainer.shap_values(ex_inputs)
        shap_values_features = shap_values_all_inputs[self._feature_input_index]
        # (n_explain, n_features)

        shap_values_features = np.array(shap_values_features)

        # Handle shapes:
        # - expected: (n_ex, n_features)
        # - you currently get: (n_ex, n_features, 1)
        if shap_values_features.ndim == 3:
          # If last dim is 1 → (n_ex, n_features, 1) → squeeze last
          if shap_values_features.shape[-1] == 1:
              shap_values_features = shap_values_features[..., 0]
          # If second dim is 1 → (n_ex, 1, n_features) → squeeze middle
          elif shap_values_features.shape[1] == 1:
              shap_values_features = shap_values_features[:, 0, :]
        # Now: (n_ex, n_features)

        # Global mean |SHAP|
        mean_abs_all = np.mean(np.abs(shap_values_features), axis=0)  # (n_features,)

        # Positive / negative groups
        pred_scores = preds[:, 0]
        pos_mask = pred_scores >= self.positive_threshold
        neg_mask = ~pos_mask

        if pos_mask.any():
            mean_abs_pos = np.mean(np.abs(shap_values_features[pos_mask]), axis=0)
        else:
            mean_abs_pos = np.zeros_like(mean_abs_all)

        if neg_mask.any():
            mean_abs_neg = np.mean(np.abs(shap_values_features[neg_mask]), axis=0)
        else:
            mean_abs_neg = np.zeros_like(mean_abs_all)

        def topk_indices(x: np.ndarray, k: int):
            # robust 1D top-k
            x = x.reshape(-1)
            k = min(k, x.shape[0])
            return np.argsort(x)[::-1][:k]

        topk_all_idx = topk_indices(mean_abs_all, self.top_k)
        topk_pos_idx = topk_indices(mean_abs_pos, self.top_k)
        topk_neg_idx = topk_indices(mean_abs_neg, self.top_k)

        stats = {
            "mean_abs_all": mean_abs_all,
            "mean_abs_pos": mean_abs_pos,
            "mean_abs_neg": mean_abs_neg,
            "topk_all_indices": topk_all_idx,
            "topk_pos_indices": topk_pos_idx,
            "topk_neg_indices": topk_neg_idx,
            "explained_indices": ex_idx,
        }

        return stats, shap_values_features, preds

from pathlib import Path
from typing import List

import numpy as np
import tensorflow as tf
from keras.callbacks import Callback

class ShapCallback(Callback):
    """
    Runs ShapEngine every N epochs on a frozen validation subset and produces:
      - grouped barplot (Global vs Pos vs Neg) for top-k raw features
      - 2x2 scatter for top-4 numeric raw features (pos vs neg predictions)

    Explanation set is built at on_train_begin by concatenating multiple
    batches from `data_source`, up to `max_samples` samples (and optionally
    up to `max_batches` batches).
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        validation_df,
        feat_cols: List[str],
        encoded_feature_names: List[str],
        base_dir: str,
        background_size: int = 256,
        max_explain_samples: int = 256,
        top_k: int = 15,
        positive_threshold: float = 0.5,
        every_n_epochs: int = 1,
        max_samples: int = 512,   # total samples in explanation set
        max_batches: int | None = None,  # limit num batches to read, or None
        rng_seed: int = 0,
    ):
        super().__init__()
        self.data_source = data_source
        self.validation_df = validation_df
        self.feat_cols = feat_cols
        self.encoded_feature_names = encoded_feature_names

        self.background_size = int(background_size)
        self.max_explain_samples = int(max_explain_samples)
        self.top_k = int(top_k)
        self.positive_threshold = float(positive_threshold)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)

        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        # frozen explanation set
        self._images = None
        self._features = None
        self._labels = None
        self._raw_batch_df = None

        # ShapEngine instance
        self._engine = ShapEngine(
            background_size=self.background_size,
            max_explain_samples=self.max_explain_samples,
            top_k=self.top_k,
            positive_threshold=self.positive_threshold,
        )

    # ---------- helpers ---------- #
    def _freeze_batch_from_dataset(self, ds: tf.data.Dataset):
        """
        Build explanation set by concatenating multiple batches from ds.

        Assumes ds yields (inputs, labels) where inputs is {'images','features'}.
        Also builds the corresponding raw_batch_df from validation_df (shuffle=False).
        """
        X_img_list, X_feat_list, y_list = [], [], []
        n_collected = 0

        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches is not None and b_idx >= self.max_batches:
                break

            if not isinstance(x, dict):
                raise ValueError("Expected dataset elements like ({'images','features'}, y).")

            images_b = x["images"].numpy()
            features_b = x["features"].numpy()
            labels_b = y.numpy()

            remaining = self.max_samples - n_collected
            if remaining <= 0:
                break

            if images_b.shape[0] > remaining:
                images_b = images_b[:remaining]
                features_b = features_b[:remaining]
                labels_b = labels_b[:remaining]

            X_img_list.append(images_b)
            X_feat_list.append(features_b)
            y_list.append(labels_b)

            n_collected += images_b.shape[0]
            if n_collected >= self.max_samples:
                break

        if n_collected == 0:
            raise RuntimeError("No samples collected for SHAP explanation set.")

        self._images = np.concatenate(X_img_list, axis=0)
        self._features = np.concatenate(X_feat_list, axis=0)
        self._labels = np.concatenate(y_list, axis=0)

        # assume validation_ds built from validation_df with shuffle=False
        self._raw_batch_df = (
            self.validation_df.iloc[:n_collected][self.feat_cols].reset_index(drop=True)
        )

    # ---------- Keras hooks ---------- #
    def on_train_begin(self, logs=None):
        self._freeze_batch_from_dataset(self.data_source)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch % self.every_n_epochs) != 0:
            return
        if self._images is None or self._features is None:
            return

        np.random.seed(self.rng_seed)

        # SHAP on frozen set with current weights
        stats, shap_vals, preds = self._engine.explain(
            validation_data=(self._features, self._images, self._labels),
            model=self.model,
            class_index=0,
        )

        mean_abs_all = stats["mean_abs_all"]
        mean_abs_pos = stats["mean_abs_pos"]
        mean_abs_neg = stats["mean_abs_neg"]

        global_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_all
        )
        pos_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_pos
        )
        neg_importance_raw = aggregate_raw_importance(
            self.encoded_feature_names, mean_abs_neg
        )

        epoch_name = f"epoch_{epoch:03d}"
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        # 1) grouped barplot: global vs pos vs neg (top-k)
        plot_topk_grouped_raw_global_pos_neg(
            global_imp=global_importance_raw,
            pos_imp=pos_importance_raw,
            neg_imp=neg_importance_raw,
            epoch_num = epoch_name,
            k=self.top_k,
            figsize=(10, 5),
            out_path=str(
                epoch_dir
                / f"shap_grouped_raw_top{self.top_k:02d}_glob_pos_neg_{epoch_name}.png"
            ),
        )

        # 2) scatter grid: top-4 numeric features
        plot_shap_scatter_top4_numeric(
            shap_vals=shap_vals,
            preds=preds,
            encoded_feature_names=self.encoded_feature_names,
            global_importance_raw=global_importance_raw,
            raw_batch_df=self._raw_batch_df,
            explained_indices=stats["explained_indices"],
            positive_threshold=self.positive_threshold,
            epoch_num = epoch_name,
            out_path=str(
                epoch_dir / f"shap_top4_numeric_grid_{epoch_name}.png"
            ),
        )

per-feature dim sizes: {'sex': 3, 'anatom_site_general': 6, 'tbp_tile_type': 3, 'tbp_lv_location': 21, 'combined_anatomical_site': 25, 'attribution': 8, 'age_approx': 32, 'tbp_lv_nevi_confidence': 1, 'clin_size_long_diam_mm': 1, 'tbp_lv_areaMM2': 1, 'tbp_lv_area_perim_ratio': 1, 'tbp_lv_color_std_mean': 1, 'tbp_lv_deltaLBnorm': 1, 'tbp_lv_minorAxisMM': 1, 'lesion_size_ratio': 1, 'lesion_shape_index': 1, 'hue_contrast': 1, 'luminance_contrast': 1, 'lesion_color_difference': 1, 'border_complexity': 1, '3d_position_distance': 1, 'perimeter_to_area_ratio': 1, 'lesion_visibility_score': 1, 'symmetry_border_consistency': 1, 'color_consistency': 1, 'size_age_interaction': 1, 'hue_color_std_interaction': 1, 'lesion_severity_index': 1, 'shape_complexity_index': 1, 'color_contrast_index': 1, 'log_lesion_area': 1, 'mean_hue_difference': 1, 'std_dev_contrast': 1, 'color_shape_composite_index': 1, '3d_lesion_orientation': 1, 'overall_color_difference': 1, 'symmetry_perimeter_interaction': 1, 'compr

#### Gradcam

In [21]:
import os
from pathlib import Path
from typing import Optional, Tuple, List, Union

import cv2
import numpy as np
import tensorflow as tf
from keras import Model


class MultiModalGradCAMEngine:
    """
    Grad-CAM engine for multimodal models (images + tabular features).

    Features:
    - Robust target selection: defaults to the 4D tensor feeding the first GlobalAveragePooling2D.
    - Two modes:
        * 'context'     → uses real tabular features.
        * 'image_only'  → replaces features with a baseline (zeros or provided vector).
    - Output:
        * grid: single grid image (HxWx3, float [0..1]) of horizontally stacked triptychs
        * triptychs: list of per-sample images (orig+pred | heatmap | overlay), each Hx(3W)x3 in [0..1]
        * preds: model predictions for the batch (numpy array)
    - Visualization:
        * Accepts a separate display_images batch or a deprocess_fn to convert model-input images to [0,1].
    """

    def __init__(
        self,
        use_guided_grads: bool = True,
        mode: str = 'context',              # 'context' or 'image_only'
        baseline: Optional[np.ndarray] = None,
        deprocess_fn: Optional[callable] = None,  # fn(img_float) -> float [0,1]
    ):
        assert mode in ('context', 'image_only'), "mode must be 'context' or 'image_only'"
        self.use_guided_grads = use_guided_grads
        self.mode = mode
        self.baseline = baseline
        self.deprocess_fn = deprocess_fn

    def explain(
        self,
        validation_data: Tuple[dict, np.ndarray],                            # ({'images', 'features'}, labels)
        model: tf.keras.Model,
        class_index: int,
        layer_name: Optional[str] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], np.ndarray]:
        """
        Compute Grad-CAMs for a batch and return (grid, triptychs, preds).

        validation_data is # ({'images', 'features'}, labels)

        """
        # --- Unpack ---
        x, labels = validation_data
        images = x['images']
        features = x['features']

        # --- Build grad model ---
        conv_target_tensor = self._resolve_target_tensor(model, layer_name)
        grad_model = Model(inputs=model.inputs, outputs=[conv_target_tensor, model.output])

        # --- Build feeds in model.inputs order ---
        # feeds = [features, images]
        # if self.mode == 'image_only':Fif se
        #     feeds = self._apply_baseline_to_feeds(model, feeds, images, features)

        feeds = self._ordered_feeds(model, images=images, features=features)
        if self.mode == 'image_only':
            feeds = self._apply_baseline_to_feeds(model, feeds, images, features)

        # --- Forward + gradients ---
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(feeds, training=False)

            # target score
            if preds.shape.rank == 2 and preds.shape[-1] == 1:
                # binary sigmoid -> use logit(p) to avoid saturation artifacts
                p = tf.clip_by_value(preds[:, 0], 1e-6, 1.0 - 1e-6)
                target = tf.math.log(p) - tf.math.log(1.0 - p)  # (B,)
            else:
                # multi-class / multi-label (keep prob target; optionally log it)
                target = preds[:, class_index]  # (B,)

            loss = tf.reduce_sum(target)  # scalar

        grads = tape.gradient(loss, conv_out)  # (B, Hc, Wc, C)

        if self.use_guided_grads:
            grads = (
                tf.cast(conv_out > 0, grads.dtype)
                * tf.cast(grads > 0, grads.dtype)
                * grads
            )

        # --- CAMs (B, Hc, Wc) ---
        cams = self._generate_cams(conv_out, grads)  # list of (Hc, Wc)
        cams = np.stack(cams, axis=0)

        # --- Triptychs with display images ---
        triptychs = []
        preds_np = preds.numpy() if isinstance(preds, tf.Tensor) else preds
        for i in range(images.shape[0]):
            pred_text = self._format_pred_text(preds_np[i])
            vis_img = images[i]
            trip = self._compose_triptych(vis_img, cams[i], pred_text)
            triptychs.append(trip)

        grid = self._make_grid(triptychs)
        return grid, triptychs, preds_np

    # ------------------------ Saving helpers ------------------------
    def save_grid(self, grid: np.ndarray, output_dir: str, output_name: str) -> None:
        """Save the grid image to disk (RGB)."""
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        out_path = str(Path(output_dir) / output_name)
        img_u8 = np.uint8(255 * np.clip(grid, 0, 1))
        cv2.imwrite(out_path, cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    def save_triptychs(
        self,
        triptychs: List[np.ndarray],
        base_dir: str,
        epoch_name: str,
        prefix: str = "sample",
    ) -> None:
        """
        Save each triptych as:
          base_dir/epoch_name/{prefix}_{i:03d}.png
        """
        out_dir = Path(base_dir) / epoch_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(triptychs):
            img_u8 = np.uint8(255 * np.clip(img, 0, 1))
            path = out_dir / f"{prefix}_{i:03d}.png"
            cv2.imwrite(str(path), cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    # ------------------------ Internal helpers ------------------------
    def _make_grid(self, triptychs: List[np.ndarray]) -> np.ndarray:
        """Create a grid image; uses tf-explain grid_display if available, else a simple grid."""
        try:
            from tf_explain.utils.display import grid_display
            thumbs = np.stack([np.uint8(255 * np.clip(t, 0, 1)) for t in triptychs], axis=0)
            canvas = grid_display(thumbs)  # returns uint8 RGB
            return canvas.astype(np.float32) / 255.0
        except Exception:
            return self._naive_grid(triptychs)


    def _resolve_target_tensor(self, model: tf.keras.Model, layer_name: Optional[str]):
        """
        Prefer the 4D tensor feeding the first GlobalAveragePooling2D.
        If layer_name is provided, use that layer's output; else fallback to last 4D layer.
        """
        if layer_name is not None:
            return model.get_layer(layer_name).output
        # Prefer the tensor feeding the first GAP
        for l in model.layers:
            if isinstance(l, tf.keras.layers.GlobalAveragePooling2D):
                return l.input
        # Fallback to last 4D tensor in the graph
        for l in reversed(model.layers):
            try:
                shp = l.output_shape
            except Exception:
                continue
            if isinstance(shp, tuple) and len(shp) == 4:
                return l.output
        raise ValueError("No suitable 4D conv tensor found for Grad-CAM target")

    def _ordered_feeds(self, model: tf.keras.Model, images, features):
        """Create a list of inputs respecting model.inputs order (rank-4 → images, rank-2 → features)."""
        feeds = []
        for kt in model.inputs:
            if len(kt.shape) == 4:
                feeds.append(images)
            elif len(kt.shape) == 2:
                feeds.append(features)
            else:
                raise ValueError(f"Unexpected input rank for {kt.name}: {kt.shape}")
        return feeds

    def _apply_baseline_to_feeds(self, model: tf.keras.Model, feeds, images, features):
        """Replace the features tensor in feeds with a baseline (zeros by default)."""
        feat_inp = next(inp for inp in model.inputs if len(inp.shape) == 2)
        F = int(feat_inp.shape[-1])
        if self.baseline is None:
            baseline = tf.zeros((tf.shape(images)[0], F), dtype=getattr(features, 'dtype', tf.float32))
        else:
            base = tf.convert_to_tensor(self.baseline)
            if base.shape.rank == 1:
                baseline = tf.broadcast_to(base[None, :], [tf.shape(images)[0], F])
            else:
                baseline = base
        feeds_out = []
        for kt, fed in zip(model.inputs, feeds):
            feeds_out.append(baseline if len(kt.shape) == 2 else fed)
        return feeds_out

    def _generate_cams(self, conv_outputs: tf.Tensor, grads: tf.Tensor) -> List[np.ndarray]:
        """Compute per-sample weighted sum of conv activations; returns list of 2D CAMs normalized to [0,1]."""
        cams = []
        conv_np = conv_outputs.numpy() if isinstance(conv_outputs, tf.Tensor) else conv_outputs
        grads_np = grads.numpy() if isinstance(grads, tf.Tensor) else grads
        B = conv_np.shape[0]
        for i in range(B):
            # exclude 1-pixel border to avoid padding artifacts
            gr = grads_np[i][1:-1, 1:-1, :]
            co = conv_np[i][1:-1, 1:-1, :]
            weights = gr.mean(axis=(0, 1))  # (C,)
            cam = (co * weights[None, None, :]).sum(axis=-1)
            cam = np.maximum(cam, 0.0)
            cam_min, cam_max = cam.min(), cam.max()
            cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
            cams.append(cam)
        return cams

    def _format_pred_text(self, pred_row):
        """Format prediction text for display: binary → p=..., multi-class → ŷ=k (p=...)."""
        try:
            arr = np.array(pred_row)
        except Exception:
            return f"p={float(pred_row):.3f}"
        if arr.ndim == 0:
            return f"p={float(arr):.3f}"
        arr = arr.ravel()
        if arr.size == 1:
            return f"p={float(arr[0]):.3f}"
        else:
            k = int(np.argmax(arr))
            p = float(arr[k])
            return f"ŷ={k} (p={p:.3f})"

    def _compose_triptych(self, image_like: np.ndarray, cam_2d: np.ndarray, pred_text: str, colormap=cv2.COLORMAP_JET, image_weight=0.8) -> np.ndarray:
        """Create a 3-panel image (orig+text | heatmap | overlay), each panel HxWx3 in [0,1]."""
        img = self._to_display_rgb(image_like)
        H, W = img.shape[:2]
        # Heatmap
        heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam_2d, (W, H))),colormap)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB) / 255.0
        # Overlay
        overlay = image_weight * img + (1.0 - image_weight) * heat
        overlay = np.clip(overlay, 0, 1)
        # Original with prediction text
        orig = (img.copy() * 255).astype(np.uint8)
        txt = pred_text if isinstance(pred_text, str) else str(pred_text)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        orig = orig.astype(np.float32) / 255.0
        return np.hstack([orig, heat, overlay])

    # ---------------- Display helpers ----------------
    def _to_display_rgb(self, image: np.ndarray) -> np.ndarray:
        """
        Convert a model-input image to float RGB in [0,1] for visualization.
        Applies custom deprocess_fn if provided; else uses heuristics.
        """
        arr = image.numpy() if tf.is_tensor(image) else np.asarray(image)

        if self.deprocess_fn is not None:
            out = self.deprocess_fn(arr)
            return np.clip(self._ensure_rgb(out), 0.0, 1.0)

        # 1) uint8 → [0,1]
        if arr.dtype == np.uint8:
            arr = arr.astype(np.float32) / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # 2) Float ranges
        arr = arr.astype(np.float32)
        amin, amax = float(arr.min()), float(arr.max())

        # true [-1,1] only if there are negatives
        if amin < 0.0 and amax <= 1.1:
            arr = (arr + 1.0) / 2.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # already [0,1]
        if 0.0 <= amin and amax <= 1.1:
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # [0,255] floats
        if amin >= -1.0 and amax > 1.5 and amax <= 260:
            arr = arr / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # fallback: per-image min-max
        rng = (amax - amin) if (amax - amin) > 1e-8 else 1.0
        arr = (arr - amin) / rng
        return np.clip(self._ensure_rgb(arr), 0, 1)

    def _ensure_rgb(self, arr: np.ndarray) -> np.ndarray:
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return arr

    def _naive_grid(self, triptychs: List[np.ndarray], cols: int = 2) -> np.ndarray:
        if not triptychs:
            return np.zeros((32, 32, 3), dtype=np.float32)
        H, W, C = triptychs[0].shape
        rows = int(np.ceil(len(triptychs) / cols))
        grid = np.ones((rows * H, cols * W, C), dtype=np.float32)
        for idx, img in enumerate(triptychs):
            r = idx // cols
            c = idx % cols
            grid[r*H:(r+1)*H, c*W:(c+1)*W] = np.clip(img, 0, 1)
        return grid

from pathlib import Path
import numpy as np
import tensorflow as tf
import cv2
from keras.callbacks import Callback

class MultiModalGradCAMCallback(Callback):
    def __init__(self, engine, probe_batch, class_index=0, layer_name=None,
                 tb_dir="./logs/gradcam", save_dir=None, tag="GradCAM/grid"):
        super().__init__()
        self.engine = engine
        self.probe_batch = probe_batch
        self.class_index = class_index
        self.layer_name = layer_name
        self.tag = tag
        self.writer = tf.summary.create_file_writer(tb_dir)
        self.save_dir = Path(save_dir) if save_dir else None
        if self.save_dir:
            self.save_dir.mkdir(parents=True, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        grid, _, _ = self.engine.explain(
            validation_data=self.probe_batch,
            model=self.model,
            class_index=self.class_index,
            layer_name=self.layer_name,
        )

        # TensorBoard
        img = tf.clip_by_value(tf.convert_to_tensor(grid, tf.float32), 0.0, 1.0)[None, ...]
        with self.writer.as_default():
            tf.summary.image(self.tag, img, step=epoch)
            self.writer.flush()

        # Disk (PNG)
        if self.save_dir:
            out = (np.clip(grid, 0, 1) * 255).astype(np.uint8)
            cv2.imwrite(str(self.save_dir / f"epoch_{epoch:03d}.png"),
                        cv2.cvtColor(out, cv2.COLOR_RGB2BGR))


#### LORE


In [22]:
!pip install git+https://github.com/kdd-lab/XAI-Lib.git

  Cloning https://github.com/kdd-lab/XAI-Lib.git to /tmp/pip-req-build-g8b9e53c
  Running command git clone --filter=blob:none --quiet https://github.com/kdd-lab/XAI-Lib.git /tmp/pip-req-build-g8b9e53c
  Resolved https://github.com/kdd-lab/XAI-Lib.git to commit 8ea5a3108443ed43215e3edacd6e925c257ebcfb
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [23]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [24]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

per-feature dim sizes: {'sex': 3, 'anatom_site_general': 6, 'tbp_tile_type': 3, 'tbp_lv_location': 21, 'combined_anatomical_site': 25, 'attribution': 8, 'age_approx': 32, 'tbp_lv_nevi_confidence': 1, 'clin_size_long_diam_mm': 1, 'tbp_lv_areaMM2': 1, 'tbp_lv_area_perim_ratio': 1, 'tbp_lv_color_std_mean': 1, 'tbp_lv_deltaLBnorm': 1, 'tbp_lv_minorAxisMM': 1, 'lesion_size_ratio': 1, 'lesion_shape_index': 1, 'hue_contrast': 1, 'luminance_contrast': 1, 'lesion_color_difference': 1, 'border_complexity': 1, '3d_position_distance': 1, 'perimeter_to_area_ratio': 1, 'lesion_visibility_score': 1, 'symmetry_border_consistency': 1, 'color_consistency': 1, 'size_age_interaction': 1, 'hue_color_std_interaction': 1, 'lesion_severity_index': 1, 'shape_complexity_index': 1, 'color_contrast_index': 1, 'log_lesion_area': 1, 'mean_hue_difference': 1, 'std_dev_contrast': 1, 'color_shape_composite_index': 1, '3d_lesion_orientation': 1, 'overall_color_difference': 1, 'symmetry_perimeter_interaction': 1, 'compr

False

In [25]:
import seaborn as sns

def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="{:.2f}", padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]


def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_lore_feature_frequency(
    freq_all: dict, freq_pos: dict, freq_neg: dict,
    k: int = 10, epoch_num=None, mode: str = '', figsize=(10, 5), out_path=None,
):
    """Barplot of rule-feature frequencies: Global / Pos / Neg predictions."""
    items = sorted(freq_all.items(), key=lambda kv: kv[1], reverse=True)[:k]
    if not items:
        return
    names = [n for n, _ in items]
    df = pd.DataFrame({
        'feature': names,
        'Global': [freq_all.get(n, 0) for n in names],
        'Pos preds': [freq_pos.get(n, 0) for n in names],
        'Neg preds': [freq_neg.get(n, 0) for n in names],
    })
    df_melt = df.melt(id_vars='feature', var_name='Class', value_name='Rule frequency')
    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt, y='feature', x='Rule frequency', hue='Class',
        palette=(sns.color_palette('bright')[4], sns.color_palette('bright')[0], sns.color_palette('bright')[3]),
        edgecolor='black',
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', padding=4, fontsize=7)
    title = f'LORE Feature Frequency in Rules (top {k})'
    if mode:
        title += f' [{mode}]'
    if epoch_num is not None:
        title += f' – {epoch_num}'
    plt.title(title)
    plt.xlabel('Rule frequency (# instances)')
    plt.ylabel('')
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


def save_lore_rules_text(rules: list, epoch_num=None, mode: str = '', out_path=None) -> str:
    """Write per-instance LORE rules to a .txt file."""
    header = 'LORE Rules'
    if mode:
        header += f' [{mode}]'
    if epoch_num is not None:
        header += f' – {epoch_num}'
    lines = [header, '=' * len(header), '']

    def _fmt(subset, label):
        lines.append(f'=== {label} ({len(subset)} instances) ===')
        for i, r in subset:
            lines.append(f'[inst {i:02d}] pred={r["pred"]:.3f} label={r["label"]}')
            lines.append(f'  RULE: {r["rule_str"]}')
            for cr in r.get('crules_str', []):
                lines.append(f'  COUNTERFACTUAL: {cr}')
            lines.append('')

    pos = [(i, r) for i, r in enumerate(rules) if r['pred'] >= 0.5]
    neg = [(i, r) for i, r in enumerate(rules) if r['pred'] < 0.5]
    _fmt(pos, 'POSITIVE PREDICTIONS')
    _fmt(neg, 'NEGATIVE PREDICTIONS')

    text = "\n".join(lines)
    if out_path:
        with open(out_path, 'w') as f:
            f.write(text)
    return text


from xailib.explainers.lore_explainer import LoreTabularExplainer
from collections import defaultdict
from typing import Dict, List, Optional


class LoreEngine:
    """
    LORE on raw (pre-FeatureSpace) tabular features.
    Categoricals are label-encoded before LORE, decoded back to strings
    in the predictor before FeatureSpace.
    """

    def __init__(
        self,
        feature_space,
        feat_cols: List[str],
        max_explain_samples: int = 12,
        positive_threshold: float = 0.5,
        lore_config: Optional[Dict] = None,
    ):
        self.feature_space = feature_space
        self.feat_cols = feat_cols
        self.max_explain_samples = int(max_explain_samples)
        self.positive_threshold = float(positive_threshold)
        self.lore_config = lore_config or {
            'neigh_type': 'random',
            'size': 300,
            'random_state': 42,
            'verbose': True,
        }
        self._image_encoder = None
        self._head_model = None
        self._predict_fn = None
        # Label encoding maps — built per explain() call from actual data
        self._label_maps = {}   # col → {string: int}
        self._inv_maps = {}     # col → {int: string}
        self._cat_cols = []

    # ── Model splitting ──────────────────────────────────────────────────
    def _build_sub_models(self, model):
        if self._image_encoder is not None:
            return
        concat_layer = next(
            l for l in model.layers if isinstance(l, keras.layers.Concatenate)
        )
        img_branch = concat_layer.input[0]
        self._image_encoder = keras.Model(
            inputs=model.input["images"], outputs=img_branch, name="image_encoder",
        )
        emb_in = keras.Input(shape=(int(img_branch.shape[-1]),), name="cached_img_emb")
        feat_in = keras.Input(shape=(int(model.input["features"].shape[-1]),), name="cached_features")
        x = feat_in
        for name in ("dense", "dense_1", "dropout_1"):
            layer = model.get_layer(name)
            x = layer(x, training=False) if "dropout" in name else layer(x)
        merged = keras.layers.Concatenate()([emb_in, x])
        pred = model.get_layer("dense_2")(merged)
        self._head_model = keras.Model(inputs=[emb_in, feat_in], outputs=pred, name="head_model")
        self._predict_fn = tf.function(
            lambda embs, enc: self._head_model([embs, enc], training=False),
            reduce_retracing=True,
        )

    # ── Label encoding ───────────────────────────────────────────────────
    def _label_encode_df(self, df):
        """Label-encode categorical columns to integers. Returns encoded copy."""
        # self._cat_cols = [c for c in df.columns if df[c].dtype == 'object']
        self._cat_cols = [c for c in df.columns if c != 'target'
                          and not pd.api.types.is_numeric_dtype(df[c])]
        self._label_maps, self._inv_maps = {}, {}
        out = df.copy()
        for col in self._cat_cols:
            uniq = sorted(df[col].dropna().unique())
            lm = {v: i for i, v in enumerate(uniq)}
            self._label_maps[col] = lm
            self._inv_maps[col] = {i: v for v, i in lm.items()}
            out[col] = df[col].map(lm).astype('float64')
        return out

    # ── Predictor ────────────────────────────────────────────────────────
    def _make_predictor(self, image_embedding, feat_cols_no_target):
        head = self._head_model
        fs = self.feature_space
        emb_const = tf.constant(image_embedding[np.newaxis], dtype=tf.float32)
        cat_cols = self._cat_cols
        inv_maps = self._inv_maps
        predict_fn = self._predict_fn

        def predict(X) -> np.ndarray:
            if isinstance(X, pd.DataFrame):
                df = X.copy()
                cols = [c for c in df.columns if c != 'target']
            else:
                arr = np.asarray(X, dtype=np.float32)
                if arr.ndim == 1:
                    arr = arr.reshape(1, -1)
                df = pd.DataFrame(arr, columns=feat_cols_no_target)
                cols = feat_cols_no_target

            # Decode label-encoded categoricals back to strings
            feat_dict = {}
            for col in cols:
                if col in cat_cols:
                    ints = df[col].values.round().astype(int).clip(0, len(inv_maps[col]) - 1)
                    strings = [inv_maps[col].get(i, inv_maps[col][0]) for i in ints]
                    feat_dict[col] = tf.constant(strings)
                else:
                    feat_dict[col] = tf.constant(df[col].values.astype(np.float32))

            encoded = fs(feat_dict)
            n = encoded.shape[0]
            embs = tf.repeat(emb_const, n, axis=0)
            return predict_fn(embs, encoded).numpy()[:, 0]

        return predict

    # ── LORE wrappers ────────────────────────────────────────────────────
    def _build_explainer(self, predictor, background_df):
        class _BB:
            def __init__(self, fn, thr):
                self.fn, self.thr = fn, thr
            def predict(self, X):
                return (self.fn(X) >= self.thr).astype(int)
            def predict_proba(self, X):
                p = self.fn(X)
                return np.column_stack([1 - p, p])

        bb = _BB(predictor, self.positive_threshold)
        df_bg = background_df.copy().reset_index(drop=True)
        df_bg['target'] = bb.predict(df_bg).astype(np.int64)
        exp = LoreTabularExplainer(bb=bb)
        exp.fit(df_bg, 'target', self.lore_config)
        return exp

    # def _explain_one(self, lore_exp, x_series):
    #     try:
    #         exp = lore_exp.explain(x_series.values)
    #         rule = exp.getRules()
    #         crules = exp.getCounterfactualRules()
    #         premise = rule.get('premise', [])
    #
    #         def fmt(p):
    #             att, op, thr = p['att'], p['op'], float(p['thr'])
    #             if att in self._cat_cols:
    #                 idx = int(round(thr))
    #                 idx = max(0, min(idx, len(self._inv_maps[att]) - 1))
    #                 label = self._inv_maps[att][idx]
    #                 readable_op = '=' if op in ('>', '>=') else '≠'
    #                 return f"{att} {readable_op} '{label}'"
    #             return f"{att} {op} {thr:.4g}"
    #
    #         rule_str = ' AND '.join(fmt(p) for p in premise)
    #         rule_features = [p['att'] for p in premise]
    #         crules_str = [
    #             ' AND '.join(fmt(p) for p in cr.get('premise', []))
    #             for cr in (crules or [])
    #         ]
    #         return {'rule_str': rule_str, 'rule_features': rule_features, 'crules_str': crules_str}
    #     except Exception as e:
    #         return {'rule_str': f'[error: {e}]', 'rule_features': [], 'crules_str': []}

    def _explain_one(self, lore_exp, x_series):
        exp = lore_exp.explain(x_series.values)
        rule = exp.getRules()
        crules = exp.getCounterfactualRules()
        premise = rule.get('premise', [])

        def fmt(p):
            att, op, thr = p['att'], p['op'], float(p['thr'])
            if att in self._cat_cols:
                idx = int(round(thr))
                idx = max(0, min(idx, len(self._inv_maps[att]) - 1))
                label = self._inv_maps[att][idx]
                readable_op = '=' if op in ('>', '>=') else '≠'
                return f"{att} {readable_op} '{label}'"
            return f"{att} {op} {thr:.4g}"

        rule_str = ' AND '.join(fmt(p) for p in premise)
        rule_features = [p['att'] for p in premise]
        crules_str = [
            ' AND '.join(fmt(p) for p in cr.get('premise', []))
            for cr in (crules or [])
        ]
        return {'rule_str': rule_str, 'rule_features': rule_features, 'crules_str': crules_str}

    # ── Public API ───────────────────────────────────────────────────────
    def explain(self, raw_df, images, labels, model, mode='context'):
        self._build_sub_models(model)

        # Label-encode categoricals → all-numeric for LORE
        encoded_df = self._label_encode_df(raw_df)
        feat_cols_no_target = [c for c in encoded_df.columns if c != 'target']

        n = len(encoded_df)
        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.sort(np.random.choice(n, self.max_explain_samples, replace=False))

        ex_df = encoded_df.iloc[ex_idx].reset_index(drop=True)
        ex_labels = labels[ex_idx].ravel()

        if mode == 'context':
            img = images.mean(axis=0, keepdims=True).astype(np.float32)
        elif mode == 'tabular_only':
            img = np.zeros((1, *images.shape[1:]), dtype=np.float32)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        image_embedding = self._image_encoder(img, training=False).numpy()[0]
        predictor = self._make_predictor(image_embedding, feat_cols_no_target)
        preds = predictor(ex_df)
        lore_exp = self._build_explainer(predictor, encoded_df)

        freq_all, freq_pos, freq_neg = defaultdict(int), defaultdict(int), defaultdict(int)
        rules = []

        for i in range(len(ex_df)):
            result = self._explain_one(lore_exp, ex_df.iloc[i])
            print(f"[LORE {i+1}/{len(ex_df)}] rule_features={result['rule_features']}")
            result['pred'] = float(preds[i])
            result['label'] = int(ex_labels[i])
            rules.append(result)
            for feat in result['rule_features']:
                raw_feat = feat.split("=")[0] if "=" in feat else feat
                freq_all[raw_feat] += 1
                (freq_pos if preds[i] >= self.positive_threshold else freq_neg)[raw_feat] += 1

        return {
            'feature_freq_all': dict(freq_all),
            'feature_freq_pos': dict(freq_pos),
            'feature_freq_neg': dict(freq_neg),
            'explained_indices': ex_idx,
        }, rules, preds


    from pathlib import Path


class LoreCallback(keras.callbacks.Callback):
    """
    Runs LoreEngine every N epochs on a frozen validation subset.
    Operates on RAW features (pre-FeatureSpace) — XAI-Lib handles
    categorical encoding internally.
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        raw_feature_df: pd.DataFrame,
        feat_cols: List[str],
        feature_space,
        base_dir: str,
        max_explain_samples: int = 12,
        positive_threshold: float = 0.5,
        top_k: int = 10,
        every_n_epochs: int = 2,
        max_samples: int = 256,
        max_batches: Optional[int] = None,
        lore_config: Optional[Dict] = None,
        rng_seed: int = 0,
    ):
        super().__init__()
        self.data_source = data_source
        self.raw_feature_df = raw_feature_df  # pre-FeatureSpace validation DataFrame
        self.feat_cols = feat_cols
        self.top_k = int(top_k)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        self._images = None
        self._raw_df = None
        self._labels = None

        self._engine = LoreEngine(
            feature_space=feature_space,
            feat_cols=feat_cols,
            max_explain_samples=int(max_explain_samples),
            positive_threshold=float(positive_threshold),
            lore_config=lore_config,
        )

    def _freeze_batch(self, ds: tf.data.Dataset):
        """Collect images and labels from dataset, raw features from DataFrame."""
        imgs, labels = [], []
        n = 0
        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches and b_idx >= self.max_batches:
                break
            remaining = self.max_samples - n
            if remaining <= 0:
                break
            imgs.append(x['images'].numpy()[:remaining])
            labels.append(y.numpy()[:remaining])
            n += imgs[-1].shape[0]

        if n == 0:
            raise RuntimeError('No samples collected for LORE.')
        self._images = np.concatenate(imgs)
        self._labels = np.concatenate(labels)
        # Take corresponding rows from raw DataFrame (same order, no shuffle in val)
        self._raw_df = self.raw_feature_df.iloc[:n][self.feat_cols].reset_index(drop=True)

    def on_train_begin(self, logs=None):
        self._freeze_batch(self.data_source)

    def on_epoch_end(self, epoch: int, logs=None):
        if epoch % self.every_n_epochs != 0 or self._images is None:
            return

        np.random.seed(self.rng_seed)
        epoch_name = f'epoch{epoch:03d}'
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        for mode in ('context', 'tabular_only'):
            mode_dir = epoch_dir / mode
            mode_dir.mkdir(parents=True, exist_ok=True)

            stats, rules, preds = self._engine.explain(
                raw_df=self._raw_df,
                images=self._images,
                labels=self._labels,
                model=self.model,
                mode=mode,
            )

            plot_lore_feature_frequency(
                freq_all=stats['feature_freq_all'],
                freq_pos=stats['feature_freq_pos'],
                freq_neg=stats['feature_freq_neg'],
                k=self.top_k,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_feature_freq_top{self.top_k:02d}_{mode}_{epoch_name}.png'),
            )

            save_lore_rules_text(
                rules=rules,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_rules_{mode}_{epoch_name}.txt'),
            )

In [49]:
raise StopExecution

NameError: name 'StopExecution' is not defined

### Modality Attendance

In [26]:
# Shared runner
import os
import json
import time
import gc
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# RUN_TAG module-level RIMOSSO — generato per-call dentro run_experiment
os.makedirs(RUNS_PATH, exist_ok=True)
os.makedirs(SUMMARIES_PATH, exist_ok=True)


def _materialize_probe(n=8):
    probe_ds = build_dataset(
        validation_ids[:n],
        training_validation_hdf5,
        {k: v[:n] for k, v in validation_features.items()},
        validation_labels[:n],
        batch_size=n, shuffle=False, augment=False, cache=False,
        drop_remainder=False,
    )
    probe_ds = probe_ds.map(
        lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y)
    )
    xb, yb = next(iter(probe_ds))
    return xb, yb


class RunLogger(keras.callbacks.Callback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = csv_path
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.rows.append({
            "epoch": int(epoch),
            "loss": float(logs.get("loss", np.nan)),
            "val_loss": float(logs.get("val_loss", np.nan)),
            "auc": float(logs.get("auc", np.nan)),
            "val_auc": float(logs.get("val_auc", np.nan)),
        })
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)


def make_train_callbacks(run_dir, model_name):
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"best_{model_name}.keras")
    ckpt_cb = keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="val_auc", save_best_only=True,
        save_weights_only=False, mode="max",
    )
    callbacks = [ckpt_cb, RunLogger(os.path.join(run_dir, "epoch_metrics.csv"))]
    if "lr_cb" in globals():
        callbacks.insert(0, lr_cb)
    return callbacks, ckpt_path


def make_xai_callbacks(run_dir, model_name):
    xai_dir = os.path.join(run_dir, "xai")
    shap_dir = os.path.join(xai_dir, "shap")
    gradcam_dir = os.path.join(xai_dir, "gradcam")
    lore_dir = os.path.join(xai_dir, "lore")
    os.makedirs(shap_dir, exist_ok=True)
    os.makedirs(gradcam_dir, exist_ok=True)
    os.makedirs(lore_dir, exist_ok=True)

    global x_probe, y_probe
    if "x_probe" not in globals() or x_probe is None:
        x_probe, y_probe = _materialize_probe(n=8)

    shap_cb = ShapCallback(
        data_source=validation_ds, validation_df=validation_df,
        feat_cols=FEAT_COLS, encoded_feature_names=encoded_feature_names,
        base_dir=shap_dir, background_size=64, max_explain_samples=128,
        top_k=10, positive_threshold=0.5, every_n_epochs=1,
        max_samples=128, max_batches=5, rng_seed=0,
    )
    gm_cb = MultiModalGradCAMCallback(
        engine=MultiModalGradCAMEngine(use_guided_grads=True, mode="image_only"),
        probe_batch=(x_probe, y_probe), class_index=0,
        save_dir=gradcam_dir, tag=f"{model_name}/GradCAM",
    )
    lore_cb = LoreCallback(
        data_source=validation_ds, raw_feature_df=validation_df,
        feat_cols=FEAT_COLS, feature_space=feature_space, base_dir=lore_dir,
        max_explain_samples=6, positive_threshold=0.5, top_k=10,
        every_n_epochs=1, max_samples=256, max_batches=2,
        lore_config={'neigh_type': 'random', 'size': 300, 'random_state': 42},
        rng_seed=0,
    )
    return [shap_cb, gm_cb, lore_cb]


def run_xai_on_best_epoch(best_model, xai_callbacks, best_epoch, best_val_auc):
    logs = {"val_auc": float(best_val_auc)}
    for cb in xai_callbacks:
        if hasattr(cb, "every_n_epochs"):
            cb.every_n_epochs = 1   # forza
        cb.set_model(best_model)
        cb.set_params({"epochs": int(best_epoch + 1), "steps": None, "verbose": 0})
        if hasattr(cb, "on_train_begin"):
            cb.on_train_begin(logs={})
        cb.on_epoch_end(int(best_epoch), logs=logs)
        if hasattr(cb, "on_train_end"):
            cb.on_train_end(logs={})


def _format_settings(extra_meta):
    if not extra_meta:
        return "none"
    parts = []
    for k, v in extra_meta.items():
        parts.append(f"{k}={v:.6g}" if isinstance(v, float) else f"{k}={v}")
    return ", ".join(parts)


def _make_run_tag(setting_tag=None, epochs=None):
    """Timestamp + opzionale setting tag + epochs → folder name leggibile."""
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    parts = [ts]
    if setting_tag:
        parts.append(setting_tag)
    if epochs is not None:
        parts.append(f"e{int(epochs)}")
    return "__".join(parts)


def run_experiment(
    impl_name,
    build_model_fn,
    enable_xai,
    epochs,
    setting_tag=None,        # opzionale, finisce nel folder name e config
    extra_meta=None,
    seed=None,
):
    keras.backend.clear_session()
    gc.collect()

    run_seed = int(getattr(CFG, "seed", 42) if seed is None else seed)
    tf.keras.utils.set_random_seed(run_seed)

    run_tag = _make_run_tag(setting_tag=setting_tag, epochs=epochs)
    run_dir = os.path.join(RUNS_PATH, impl_name, run_tag)
    os.makedirs(run_dir, exist_ok=True)

    model = build_model_fn(run_eagerly=False)
    model_name = f"{impl_name}__{setting_tag}" if setting_tag else impl_name

    print("=" * 110)
    print(f"[RUN START] impl={impl_name} | tag={setting_tag or '-'} | run_tag={run_tag} "
          f"| xai={bool(enable_xai)} | epochs={epochs} | seed={run_seed}")
    print("-" * 110)

    cfg = {
        "impl_name": impl_name,
        "setting_tag": setting_tag,
        "run_tag": run_tag,
        "model_name": model_name,
        "epochs": int(epochs),
        "batch_size": int(CFG.batch_size),
        "seed": run_seed,
        "enable_xai": bool(enable_xai),
    }
    if extra_meta is not None:
        cfg.update(extra_meta)

    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    callbacks, ckpt_path = make_train_callbacks(run_dir, model_name)

    t0 = time.time()
    history = model.fit(
        training_ds, epochs=epochs, callbacks=callbacks,
        validation_data=validation_ds, verbose=CFG.verbose,
        class_weight=class_weights if "class_weights" in globals() else None,
    )
    train_seconds = time.time() - t0

    val_auc_hist = history.history.get("val_auc", [])
    if len(val_auc_hist) > 0:
        val_auc_arr = np.asarray(val_auc_hist, dtype=np.float32)
        best_epoch = int(np.nanargmax(val_auc_arr))
        best_val_auc = float(np.nanmax(val_auc_arr))
    else:
        best_epoch = -1
        best_val_auc = float("nan")

    xai_done = False
    if bool(enable_xai) and best_epoch >= 0:
        # NESSUN try/except: errori esplodono
        if os.path.exists(ckpt_path):
            best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
        else:
            best_model = model
        xai_callbacks = make_xai_callbacks(run_dir, model_name)
        if len(xai_callbacks) > 0:
            run_xai_on_best_epoch(best_model, xai_callbacks, best_epoch, best_val_auc)
            xai_done = True

    cfg.update({
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "xai_best_epoch_only": True,
        "xai_done": bool(xai_done),
    })
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history.history, f, indent=2)

    summary_df = pd.DataFrame([{
        "impl_name": impl_name,
        "setting_tag": setting_tag,
        "run_tag": run_tag,
        "seed": run_seed,
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "train_seconds": float(train_seconds),
        "xai_done": bool(xai_done),
        "run_dir": run_dir,
        "model_name": model_name,
        **(extra_meta or {}),
    }])
    summary_df.to_csv(os.path.join(run_dir, "single_run_summary.csv"), index=False)

    print("-" * 110)
    print(f"[RUN END]| best_epoch={best_epoch} | best_val_auc={best_val_auc:.6f} "
          f"| xai_done={xai_done} | time={train_seconds:.1f}s")
    print("=" * 110)

    keras.backend.clear_session()
    gc.collect()
    return summary_df, run_dir


def run_model_experiment(
    impl_name, settings, build_model_from_setting_fn, enable_xai_fn,
    epochs, extra_meta_fn=None, seed=None,
):
    rows = []
    histories = {}
    exp_tag = datetime.now().strftime("%Y%m%d-%H%M%S")
    summary_dir = os.path.join(SUMMARIES_PATH, impl_name, exp_tag)
    os.makedirs(summary_dir, exist_ok=True)

    print(f"\n[EXPERIMENT START] impl={impl_name} | n_settings={len(settings)} | exp_tag={exp_tag}")

    for i, setting in enumerate(settings, 1):
        tag = setting["tag"]
        meta = extra_meta_fn(setting) if extra_meta_fn is not None else {}
        print(f"\n[SETTING {i}/{len(settings)}] tag={tag} | xai={bool(enable_xai_fn(setting))} | settings={_format_settings(meta)}")
        summary_df, run_dir = run_experiment(
            impl_name=impl_name,
            build_model_fn=lambda run_eagerly, s=setting: build_model_from_setting_fn(s, run_eagerly),
            enable_xai=bool(enable_xai_fn(setting)),
            epochs=epochs,
            setting_tag=tag,
            extra_meta=meta,
            seed=seed,
        )
        rows.append(summary_df)
        hist_path = os.path.join(run_dir, "history.json")
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                histories[tag] = json.load(f)

    full_summary = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if not full_summary.empty:
        full_summary.to_csv(os.path.join(summary_dir, "summary.csv"), index=False)

    print(f"\n[EXPERIMENT END] impl={impl_name} | completed_settings={len(rows)}")
    return full_summary, histories, summary_dir

In [27]:
# plot runner helpers

def _finalize_plot(out_dir=None, fname=None):
    plt.tight_layout()
    if out_dir and fname:
        os.makedirs(out_dir, exist_ok=True)
        plt.savefig(os.path.join(out_dir, fname), dpi=300)
    plt.show()
    plt.close()

def plot_param_vs_best_auc(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "best_val_auc" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["best_val_auc"], marker="o")
    plt.xlabel(param_col); plt.ylabel("best_val_auc")
    plt.title(title or f"Best val AUC vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_best_val_auc.png")

def plot_param_vs_best_epoch(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "best_epoch" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["best_epoch"], marker="o")
    plt.xlabel(param_col); plt.ylabel("best_epoch")
    plt.title(title or f"Best epoch vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_best_epoch.png")

def plot_param_vs_train_time(df, param_col, title=None, out_dir=None):
    if df.empty or param_col not in df.columns or "train_seconds" not in df.columns:
        return
    d = df.sort_values(param_col)
    plt.figure(figsize=(7, 4))
    plt.plot(d[param_col], d["train_seconds"], marker="o")
    plt.xlabel(param_col); plt.ylabel("train_seconds")
    plt.title(title or f"Train time vs {param_col}")
    plt.grid(alpha=0.3)
    _finalize_plot(out_dir, f"{param_col}_vs_train_seconds.png")

def plot_val_auc_vs_epoch_sweep(df, histories, setting_col, title=None, out_dir=None, fname=None):
    if df.empty or "run_id" not in df.columns or setting_col not in df.columns or not histories:
        return

    d = df[["run_id", setting_col]].drop_duplicates().sort_values(setting_col)
    plt.figure(figsize=(8, 5))

    plotted = 0
    for _, r in d.iterrows():
        y = histories.get(r["run_id"], {}).get("val_auc", [])
        if len(y) == 0:
            continue
        x = np.arange(1, len(y) + 1)
        plt.plot(x, y, marker="o", linewidth=1.6, label=f"{setting_col}={r[setting_col]}")
        plotted += 1

    if plotted == 0:
        plt.close()
        return

    plt.xlabel("epoch")
    plt.ylabel("val_auc")
    plt.title(title or f"val_auc vs epoch by {setting_col}")
    plt.grid(alpha=0.3)
    plt.legend(loc="best")
    _finalize_plot(out_dir, fname or f"val_auc_vs_epoch_by_{setting_col}.png")

In [ ]:
raise StopExecution

### Vanilla (No explicit modality adjustment)

In [29]:
def build_vanilla(optimizer, loss, metric, run_eagerly=False):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    concat = keras.layers.Concatenate()([x1, x2])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.models.Model(inp, out)

    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


# Vanilla run
EPOCHS = 12
ENABLE_XAI = True

summary_df, run_dir = run_experiment(
    impl_name="vanilla",
    build_model_fn=lambda run_eagerly: build_vanilla(adam, bce, auc, run_eagerly),
    enable_xai=ENABLE_XAI,
    epochs=EPOCHS,
    # setting_tag="baseline",   # optional
)
print(summary_df)

[RUN START] impl=vanilla | tag=- | run_tag=20260430-224204__e12 | xai=True | epochs=12 | seed=48
--------------------------------------------------------------------------------------------------------------
Epoch 1/12
35/35 - 182s - 5s/step - auc: 0.5843 - loss: 0.6824 - val_auc: 0.7206 - val_loss: 0.6416 - learning_rate: 2.5000e-05
Epoch 2/12
35/35 - 16s - 453ms/step - auc: 0.8374 - loss: 0.5298 - val_auc: 0.8642 - val_loss: 0.5218 - learning_rate: 2.3000e-04
Epoch 3/12
35/35 - 16s - 456ms/step - auc: 0.9521 - loss: 0.3184 - val_auc: 0.9113 - val_loss: 0.3874 - learning_rate: 4.3500e-04
Epoch 4/12
35/35 - 15s - 423ms/step - auc: 0.9862 - loss: 0.1889 - val_auc: 0.9056 - val_loss: 0.4970 - learning_rate: 6.4000e-04
Epoch 5/12
35/35 - 15s - 429ms/step - auc: 0.9967 - loss: 0.1243 - val_auc: 0.9029 - val_loss: 0.5347 - learning_rate: 4.8200e-04
Epoch 6/12
35/35 - 15s - 432ms/step - auc: 0.9993 - loss: 0.0896 - val_auc: 0.8927 - val_loss: 0.6175 - learning_rate: 3.6350e-04
Epoch 7/12
35/

### Fixed modality scaling — simple approach
After obtaining embeddings **x1** (images) and **x2** (tabular), we scale them with **β = 1 − α** and **α**, respectively.

- If **α = 0** → model uses only images  
- If **α = 0.5** → balanced usage  
- If **α = 1** → model uses only tabular


In [ ]:
# Fixed scaling ---------------------------------------------------------------
@keras.utils.register_keras_serializable(package="modality_attendance")
class ConstantScale(keras.layers.Layer):
    def __init__(self, scale, **kwargs):
        super().__init__(**kwargs)
        self.scale = float(scale)

    def call(self, x):
        return x * keras.ops.cast(self.scale, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"scale": self.scale})
        return cfg


def build_fixed_scaling(alpha, optimizer, loss, metric, run_eagerly=False):
    alpha = float(alpha)
    beta = 1.0 - alpha

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    x1_weighted = ConstantScale(beta, name="image_weighting")(x1)
    x2_weighted = ConstantScale(alpha, name="tabular_weighting")(x2)

    concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.Model(inp, out, name="fixed_alpha")
    opt = keras.optimizers.Adam.from_config(optimizer.get_config())
    model.compile(
        optimizer=opt,
        loss=loss,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


# ALPHAS = sorted({0.0, 0.5, 1.0})
# XAI_ALPHAS = {0.5}
# EPOCHS = 1
SEED = 49

FIXED_SETTINGS = [{"tag": f"a{a:.3f}".replace(".", "p"), "alpha": float(a)} for a in ALPHAS]

fixed_results, _, fixed_summary_dir = run_model_experiment(
    impl_name="fixed_scaling",
    settings=FIXED_SETTINGS,
    build_model_from_setting_fn=lambda s, run_eagerly: build_fixed_scaling(
        s["alpha"], adam, bce, auc, run_eagerly
    ),
    enable_xai_fn=lambda s: round(float(s["alpha"]), 3) in XAI_ALPHAS,
    epochs=EPOCHS,
    extra_meta_fn=lambda s: {"alpha": float(s["alpha"]), "beta": float(1.0 - s["alpha"])},
    seed=SEED,
)

display(
    fixed_results[
        ["impl_name", "run_id", "alpha", "best_val_auc", "best_epoch", "train_seconds", "seed", "xai_done"]
    ]
)

# Save cross-setting plots alongside the summary CSV
plots_dir = os.path.join(fixed_summary_dir, "plots")
plot_param_vs_best_auc(fixed_results, "alpha", out_dir=plots_dir)
plot_param_vs_best_epoch(fixed_results, "alpha", out_dir=plots_dir)
plot_param_vs_train_time(fixed_results, "alpha", out_dir=plots_dir)

print("[plots]:", plots_dir)

### Learnable modality scaling — symmetric (ablation)

We learn a single trainable logit `w` and compute:

- `alpha = sigmoid(w)` with `alpha ∈ [0,1]`
- `beta = 1 - alpha` (symmetry constraint)

`init_alpha` sets the starting balance (typically `0.5`).  

During training, backprop updates `w`, so the model can shift attention between modalities while keeping total modality weight fixed.

- `alpha → 0`: image-dominant
- `alpha ≈ 0.5`: balanced
- `alpha → 1`: tabular-dominant

For each run, report both `init_alpha` and learned `alpha_final` (from the best checkpoint).

In [ ]:
# Learnable modality scaling (init-alpha sweep)

@keras.utils.register_keras_serializable(package="modality_attendance")
class LearnableAlpha(keras.layers.Layer):
    def __init__(self, init_alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        init_alpha = float(np.clip(init_alpha, 1e-4, 1.0 - 1e-4))
        self.init_alpha = init_alpha
        init_logit = np.log(init_alpha / (1.0 - init_alpha))
        self.logit = self.add_weight(
            name="alpha_logit",
            shape=(),
            initializer=keras.initializers.Constant(init_logit),
            trainable=True,
        )

    def call(self, x):
        alpha = tf.sigmoid(self.logit)
        beta = 1.0 - alpha
        return alpha, beta

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"init_alpha": self.init_alpha})
        return cfg


@keras.utils.register_keras_serializable(package="modality_attendance")
class ScalarMultiply(keras.layers.Layer):
    def call(self, inputs):
        x, s = inputs
        s = keras.ops.cast(s, x.dtype)
        return x * s

    def compute_output_shape(self, input_shape):
        return input_shape[0]


def build_learnable_scaling(setting, run_eagerly=False):
    init_alpha = float(setting.get("init_alpha", 0.5))

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    alpha, beta = LearnableAlpha(init_alpha=init_alpha, name="learnable_alpha")(x1)
    x1_weighted = ScalarMultiply(name="image_weighting")([x1, beta])
    x2_weighted = ScalarMultiply(name="tabular_weighting")([x2, alpha])

    concat = keras.layers.Concatenate(name="concat")([x1_weighted, x2_weighted])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

    model = keras.Model(inp, out, name="learnable_scaling")

    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


def read_alpha_from_best_ckpt(row):
    ckpt_path = os.path.join(row["run_dir"], "checkpoints", f"best_{row['model_name']}.keras")
    if not os.path.exists(ckpt_path):
        return np.nan
    m = keras.models.load_model(ckpt_path, compile=False)
    a_layer = m.get_layer("learnable_alpha")
    return float(tf.sigmoid(a_layer.logit).numpy())


# Learnable modality scaling
INIT_ALPHAS = sorted({0.0, 0.5, 1.0})
# INIT_ALPHAS = np.round(np.arange(0.0, 1.001, 0.125), 3)
XAI_INIT_ALPHAS = {0.5}
# XAI_INIT_ALPHAS = set(np.round(np.arange(0.125, 1.0, 0.125), 3).tolist())
EPOCHS = 1

LEARNABLE_SETTINGS = [
    {"tag": f"init_a{a:.3f}".replace(".", "p"), "init_alpha": float(a)}
    for a in INIT_ALPHAS
]

SEED = 49

learnable_results, learnable_histories, learnable_summary_dir = run_model_experiment(
    impl_name="learnable_scaling",
    settings=LEARNABLE_SETTINGS,
    build_model_from_setting_fn=build_learnable_scaling,
    enable_xai_fn=lambda s: round(float(s["init_alpha"]), 3) in XAI_INIT_ALPHAS,
    epochs=EPOCHS,
    extra_meta_fn=lambda s: {"init_alpha": float(s["init_alpha"])},
    seed=SEED,
)

learnable_results["alpha_final"] = learnable_results.apply(read_alpha_from_best_ckpt, axis=1)
learnable_results["beta_final"] = 1.0 - learnable_results["alpha_final"]

display(
    learnable_results[
        [
            "impl_name", "run_id", "init_alpha", "alpha_final", "beta_final",
            "best_epoch", "best_val_auc", "train_seconds", "xai_done", "seed",
        ]
    ]
)

# Standard framework plots
learnable_plots_dir = os.path.join(learnable_summary_dir, "plots")
plot_param_vs_best_auc(
    learnable_results, "init_alpha",
    title="Learnable scaling: best val AUC vs init_alpha",
    out_dir=learnable_plots_dir,
)
plot_param_vs_best_epoch(
    learnable_results, "init_alpha",
    title="Learnable scaling: best epoch vs init_alpha",
    out_dir=learnable_plots_dir,
)
plot_param_vs_train_time(
    learnable_results, "init_alpha",
    title="Learnable scaling: train time vs init_alpha",
    out_dir=learnable_plots_dir,
)

# Epoch curves (one curve per init_alpha)
plot_val_auc_vs_epoch_sweep(
    learnable_results,
    learnable_histories,
    setting_col="init_alpha",
    title="Learnable scaling: val_auc vs epoch by init_alpha",
    out_dir=learnable_plots_dir,
    fname="val_auc_vs_epoch_init_alpha_sweep.png",
)

# init_alpha vs alpha_final
d = learnable_results.sort_values("init_alpha")
plt.figure(figsize=(7, 4))
plt.plot(d["init_alpha"], d["alpha_final"], marker="o", label="alpha_final")
plt.plot(
    [d["init_alpha"].min(), d["init_alpha"].max()],
    [d["init_alpha"].min(), d["init_alpha"].max()],
    "--", linewidth=1, label="y=x",
)
plt.xlabel("init_alpha")
plt.ylabel("alpha_final")
plt.title("Learnable scaling: initialization sensitivity")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
os.makedirs(learnable_plots_dir, exist_ok=True)
plt.savefig(os.path.join(learnable_plots_dir, "init_alpha_vs_alpha_final.png"), dpi=300)
plt.show()
plt.close()

print("[plots]:", learnable_plots_dir)

### Loss regularization -- Refined approach

Here we do not modify the architecture, but add a penalty term to the loss that reduces the dependence on the tabular (or image) branch.

We measure the "dependence" through the L2 norm of the weights connecting the modality to the final head.

We apply modality-aware L2 regularization on final-layer modality-specific weights.

- **Per-sample mode**:  
  `L_i = BCE_i + λ_reg * R`
- **Per-batch mode**:  
  `L = mean(BCE_i) + λ_reg * R`

Both use the same regularizer `R`, but per-batch gives a single scalar objective per step and is usually more stable for logging/comparison.

In [ ]:
# per sample reg
def weighted_loss_ps(y_true, y_pred):
    # BCE base
    base_loss = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)

    # Penalità sui pesi che connettono la parte tabellare all’output
    W = dense_final.kernel  # matrice [dim_img + dim_tab, 1]
    W_tab = W[dim_img:]     # seleziona solo la parte tabellare
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))

    return base_loss + reg_term

# per batch reg
def weighted_loss_pb(y_true, y_pred):
    base_loss = keras.losses.binary_crossentropy(
        y_true, y_pred, label_smoothing=0.02
    )
    base_loss = tf.reduce_mean(base_loss)  # per-batch mean
    W = dense_final.kernel
    W_tab = W[dim_img:]
    reg_term = lambda_reg * tf.reduce_sum(tf.square(W_tab))
    return base_loss + reg_term


def build_loss_regularized(
    lambda_reg=1e-4,
    penalty_on="tabular",          # "tabular" | "image" | "both"
    reg_mode="per_batch",          # "per_sample" | "per_batch"
    lambda_sym=1e-3,               # symmetry strength
    sym_target_tab=0.5,            # 0.5 => equal image/tab influence
    run_eagerly=False,
):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    concat = keras.layers.Concatenate(name="concat")([x1, x2])
    out_layer = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="output")
    out = out_layer(concat)
    model = keras.Model(inp, out, name="loss_regularization")

    dim_img = int(x1.shape[-1])
    dim_tab = int(x2.shape[-1])

    def _modality_norms():
        W = out_layer.kernel  # [dim_img + dim_tab, 1]
        W_img = W[:dim_img]
        W_tab = W[dim_img:dim_img + dim_tab]
        # mean square (not sum) avoids bias from different branch dims
        n_img = tf.reduce_mean(tf.square(W_img))
        n_tab = tf.reduce_mean(tf.square(W_tab))
        return n_img, n_tab

    def _reg_term():
        n_img, n_tab = _modality_norms()

        if penalty_on == "tabular":
            mag_term = lambda_reg * n_tab
        elif penalty_on == "image":
            mag_term = lambda_reg * n_img
        elif penalty_on == "both":
            mag_term = lambda_reg * (n_img + n_tab)
        else:
            raise ValueError(f"Unsupported penalty_on={penalty_on}")

        # soft symmetry: push tab share toward target (0.5 = symmetric)
        tab_share = n_tab / (n_img + n_tab + 1e-8)
        sym_term = lambda_sym * tf.square(tab_share - sym_target_tab)

        return mag_term + sym_term

    def weighted_loss_ps(y_true, y_pred):
        base = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)
        return base + _reg_term()

    def weighted_loss_pb(y_true, y_pred):
        base = keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.02)
        base = tf.reduce_mean(base)
        return base + _reg_term()

    loss_fn = weighted_loss_ps if reg_mode == "per_sample" else weighted_loss_pb

    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=loss_fn,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

# regularized run
REG_MODE = "per_batch"
LAMBDA_REG = 1e-4
PENALTY_ON = "tabular"
EPOCHS = 8
SEED = 49

RUN_ID = f"{REG_MODE}_lam{LAMBDA_REG:.0e}_{PENALTY_ON}"

reg_summary, reg_run_dir = run_experiment(
    impl_name="loss_regularization",
    build_model_fn=lambda run_eagerly: build_loss_regularized(
        lambda_reg=LAMBDA_REG,
        penalty_on=PENALTY_ON,
        reg_mode=REG_MODE,
        run_eagerly=run_eagerly,
    ),
    enable_xai=True,
    epochs=EPOCHS,
    extra_meta={
        "reg_mode": REG_MODE,
        "lambda_reg": float(LAMBDA_REG),
        "penalty_on": PENALTY_ON,
    },
    seed=SEED,
)

display(
    reg_summary[
        ["impl_name", "run_id", "best_val_auc", "best_epoch", "train_seconds", "seed", "xai_done", "reg_mode", "lambda_reg", "penalty_on"]
    ]
)

# Load history for epoch plots
with open(os.path.join(reg_run_dir, "history.json")) as f:
    reg_hist = json.load(f)

reg_plots_dir = os.path.join(reg_run_dir, "plots")
os.makedirs(reg_plots_dir, exist_ok=True)

# val_auc vs epoch
if "val_auc" in reg_hist and len(reg_hist["val_auc"]) > 0:
    x = np.arange(1, len(reg_hist["val_auc"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(x, reg_hist["val_auc"], marker="o")
    plt.xlabel("epoch")
    plt.ylabel("val_auc")
    plt.title("Loss regularization: val_auc vs epoch")
    plt.grid(alpha=0.3)
    _finalize_plot(reg_plots_dir, "val_auc_vs_epoch.png")

# loss + val_loss vs epoch
if "loss" in reg_hist and "val_loss" in reg_hist and len(reg_hist["loss"]) > 0:
    x = np.arange(1, len(reg_hist["loss"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(x, reg_hist["loss"], marker="o", label="loss")
    plt.plot(x, reg_hist["val_loss"], marker="o", label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Loss regularization: loss vs epoch")
    plt.grid(alpha=0.3)
    plt.legend()
    _finalize_plot(reg_plots_dir, "loss_vs_epoch.png")

print("[plots]:", reg_plots_dir)

### Controlled fusion (post‑training weighting)  
This variant replaces the standard merge with a weighted sum of the two modality embeddings:  
`fused = gamma_img * z_img + gamma_tab * z_tab`.  
It lets you manually bias the model toward image or tabular features *after training* by changing `gamma_img` and `gamma_tab` (both non‑trainable).  
Note: `z_img` and `z_tab` must have the same dimensionality, since they are summed.

In [ ]:
# Controlled fusion (post-training weighting)

@keras.utils.register_keras_serializable(package="modality_attendance")
class FusionWithGamma(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gamma_img = self.add_weight(name="gamma_img", shape=(), initializer="ones", trainable=False)
        self.gamma_tab = self.add_weight(name="gamma_tab", shape=(), initializer="ones", trainable=False)

    def call(self, inputs):
        z_img, z_tab = inputs
        return self.gamma_img * z_img + self.gamma_tab * z_tab


def build_gamma_fusion(setting, run_eagerly=False):
    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.Dense(128, activation="linear", name="img_proj")(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)

    fused = FusionWithGamma(name="fusion_with_gamma")([x1, x2])
    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(fused)

    model = keras.Model(inp, out, name="controlled_fusion")
    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model


# training run
CF_TAG = "base"
EPOCHS = 8
SEED = 49

cf_summary, cf_run_dir = run_experiment(
    impl_name="controlled_fusion",
    build_model_fn=lambda run_eagerly: build_gamma_fusion({"tag": CF_TAG}, run_eagerly),
    enable_xai=True,  # best epoch only, by shared runner design
    epochs=EPOCHS,
    extra_meta={},
    seed=SEED,
)

display(cf_summary)

 After training, you can manually adjust them to bias the final prediction toward one modality without retraining

In [ ]:
# Load best checkpoint
row = cf_summary.iloc[0]
ckpt_path = os.path.join(cf_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
fusion_layer = best_model.get_layer("fusion_with_gamma")

def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# gamma_tab > 1 enhances tabular / penalizes image (gamma_img < 1)
# gamma_tab < 1 penalizes tabular / enhances image (gamma_img > 1)
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

rows = []
for s in POSTHOC_SETTINGS:
    fusion_layer.gamma_img.assign(s["gamma_img"])
    fusion_layer.gamma_tab.assign(s["gamma_tab"])
    auc_val = eval_auc(best_model, validation_ds)

    rows.append({
        "posthoc_id": s["tag"],
        "delta": s["delta"],
        "gamma_img": s["gamma_img"],
        "gamma_tab": s["gamma_tab"],
        "mode": s["mode"],
        "val_auc": auc_val,
        "base_run_id": row["run_id"],
        "base_model_name": row["model_name"],
    })

posthoc_df = pd.DataFrame(rows).sort_values("gamma_tab").reset_index(drop=True)
posthoc_df.to_csv(os.path.join(cf_run_dir, "diagnostics", "posthoc_gamma_summary.csv"), index=False)
display(posthoc_df)

# Plots
plots_dir = os.path.join(cf_run_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

plt.figure(figsize=(7, 4))
plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
plt.axvline(1.0, linestyle="--", linewidth=1)
plt.xlabel("gamma_tab")
plt.ylabel("val_auc")
plt.title("Controlled fusion post-hoc: val_auc vs gamma_tab")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
plt.show()
plt.close()

plt.figure(figsize=(7, 4))
plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
for _, r in posthoc_df.iterrows():
    plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
plt.xlabel("gamma_img")
plt.ylabel("gamma_tab")
plt.title("Controlled fusion post-hoc sweep (val_auc)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
plt.show()
plt.close()


### Adaptive gating

In [ ]:
# # Per evitare che il gate diventi troppo deterministico (cioè sempre 0 o 1),
# # possiamo aggiungere un termine di entropia alla loss

# def gated_loss(y_true, y_pred):
#     base_loss = keras.losses.binary_crossentropy(y_true, y_pred,
#                                                  label_smoothing=0.02)
#     gate_out = model.get_layer("gate_weights").output
#     entropy = -tf.reduce_mean(tf.reduce_sum(gate_out * tf.math.log(gate_out + 1e-6), axis=1))
#     return base_loss + 1e-4 * entropy

In [ ]:
import keras.ops as K

@keras.utils.register_keras_serializable(package="modality_attendance")
class GateEntropyLoss(keras.layers.Layer):
    def __init__(self, weight=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.weight = float(weight)

    def call(self, gate_out):
        entropy = -K.mean(K.sum(gate_out * K.log(gate_out + 1e-6), axis=1))
        self.add_loss(-self.weight * entropy)  # maximize entropy
        return gate_out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"weight": self.weight})
        return cfg


def build_adaptive_gating(setting, run_eagerly=False):
    entropy_weight = float(setting.get("entropy_weight", 1e-4))

    image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
    feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
    inp = {"images": image_input, "features": feat_input}

    backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
    x1 = backbone(image_input)
    x1 = keras.layers.GlobalAveragePooling2D()(x1)
    x1 = keras.layers.Dropout(0.2)(x1)
    x1 = keras.layers.Dense(256, activation="selu", name="img_emb")(x1)

    x2 = keras.layers.Dense(96, activation="selu")(feat_input)
    x2 = keras.layers.Dense(128, activation="selu")(x2)
    x2 = keras.layers.Dropout(0.1)(x2)
    x2 = keras.layers.Dense(256, activation="selu", name="tab_emb")(x2)

    fusion_input = keras.layers.Concatenate(name="fusion_input")([x1, x2])
    gate = keras.layers.Dense(64, activation="relu")(fusion_input)
    gate = keras.layers.Dense(2, activation="softmax", name="gate_weights")(gate)
    gate = GateEntropyLoss(weight=entropy_weight, name="gate_entropy_loss")(gate)

    w_img = keras.layers.Lambda(lambda w: w[:, 0:1], name="w_img")(gate)
    w_tab = keras.layers.Lambda(lambda w: w[:, 1:2], name="w_tab")(gate)

    x1_w = keras.layers.Multiply(name="weighted_img")([x1, w_img])
    x2_w = keras.layers.Multiply(name="weighted_tab")([x2, w_tab])
    fused = keras.layers.Add(name="fused")([x1_w, x2_w])

    out = keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="output")(fused)

    model = keras.Model(inp, out, name="adaptive_gating")
    opt = keras.optimizers.Adam.from_config(adam.get_config())
    model.compile(
        optimizer=opt,
        loss=bce,
        metrics=[keras.metrics.AUC(name="auc")],
        run_eagerly=run_eagerly,
        jit_compile=not run_eagerly,
    )
    return model

AG_SETTING = {"tag": "base", "entropy_weight": 1e-4}
EPOCHS = 8
SEED = 49

ag_summary, ag_run_dir = run_experiment(
    impl_name="adaptive_gating",
    build_model_fn=lambda run_eagerly: build_adaptive_gating(AG_SETTING, run_eagerly),
    enable_xai=True,   # best epoch only (shared runner behavior)
    epochs=EPOCHS,
    extra_meta=AG_SETTING,
    seed=SEED,
)

display(ag_summary)


In [ ]:
with open(os.path.join(ag_run_dir, "history.json")) as f:
    ag_hist = json.load(f)

plots_dir = os.path.join(ag_run_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

if "val_auc" in ag_hist and len(ag_hist["val_auc"]) > 0:
    x = np.arange(1, len(ag_hist["val_auc"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(x, ag_hist["val_auc"], marker="o")
    plt.xlabel("epoch"); plt.ylabel("val_auc")
    plt.title("Adaptive gating: val_auc vs epoch")
    plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "val_auc_vs_epoch.png")

if "loss" in ag_hist and "val_loss" in ag_hist and len(ag_hist["loss"]) > 0:
    x = np.arange(1, len(ag_hist["loss"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(x, ag_hist["loss"], marker="o", label="loss")
    plt.plot(x, ag_hist["val_loss"], marker="o", label="val_loss")
    plt.xlabel("epoch"); plt.ylabel("loss")
    plt.title("Adaptive gating: loss vs epoch")
    plt.grid(alpha=0.3)
    plt.legend()
    _finalize_plot(plots_dir, "loss_vs_epoch.png")

#### Sanity cheks

##### Modality Ablation

Evaluates the model's reliance on each modality (image and tabular data).

It quantifies the performance drop (AUC) when a specific modality is either fully removed or partially perturbed, helping to understand its contribution to the overall prediction.

##### **Gate Faithfulness**

Assesses how well the dynamically predicted gate weights align with the actual impact of each modality on the model's output.

A higher **Spearman correlation** between a modality's gate weight and its ablation impact suggests that the gate is faithfully representing the modality's importance.

##### **Calibration (Brier + ECE)**

Evaluates the model's calibration i.e. how well its predicted probabilities match the true probabilities.

**Brier Score** measures the mean squared difference between predicted probabilities and actual outcomes, indicating overall accuracy and calibration.

**Expected Calibration Error (ECE)** quantifies how far predicted probabilities are from true correctness across different confidence bins, revealing systematic over- or under-confidence.

In [ ]:
# post-training diagnostics

# Load best model from the training run
row = ag_summary.iloc[0]
best_ckpt = os.path.join(ag_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
ag_model = keras.models.load_model(best_ckpt, compile=False, safe_mode=False)

# Generic AUC evaluator on a dataset
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Build ablated inputs to remove/perturb one modality
def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)  # gray image
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

# Evaluate AUC after modality ablation
def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(ablate_batch(x, mode, strength), training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Simple Spearman (rank correlation) implementation
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean()
    rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))

# Convert probabilities to logits for impact comparison
def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1 - eps)
    return tf.math.log(p / (1 - p))

# Gate-faithfulness: do gate weights correlate with true modality impact?
def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []
    for x, _ in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # Impact after image ablation
        pi = tf.squeeze(model(ablate_batch(x, "img", strength), training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # Impact after tabular ablation
        pt = tf.squeeze(model(ablate_batch(x, "tab", strength), training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        # Predicted gate weights
        w = gate_model(x, training=False)
        wimg.append(w[:, 0].numpy())
        wtab.append(w[:, 1].numpy())
        dimg.append(di.numpy())
        dtab.append(dt.numpy())

    wimg = np.concatenate(wimg)
    wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg)
    dtab = np.concatenate(dtab)

    return {
        "spearman_w_img_delta_img": spearman(wimg, dimg),
        "spearman_w_tab_delta_tab": spearman(wtab, dtab),
    }

# Calibration metrics: Brier score + ECE
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)
    brier = float(np.mean((p - y) ** 2))

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.any(m):
            conf = float(np.mean(p[m]))
            acc = float(np.mean(y[m]))
            ece += (np.sum(m) / N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

# Read average gate assignment on one validation batch
gate_model = keras.Model(ag_model.inputs, ag_model.get_layer("gate_weights").output)
gx, _ = next(iter(validation_ds))
gw = gate_model(gx, training=False).numpy()

# Modality ablation summary
base_auc = eval_auc(ag_model, validation_ds)
img_auc = eval_auc_with_ablation(ag_model, validation_ds, "img", 1.0)
tab_auc = eval_auc_with_ablation(ag_model, validation_ds, "tab", 1.0)

# Aggregate all diagnostics
diag = {
    "gate_weight_mean_img": float(np.mean(gw[:, 0])),
    "gate_weight_mean_tab": float(np.mean(gw[:, 1])),
    "base_auc": base_auc,
    "img_drop_auc": base_auc - img_auc,
    "tab_drop_auc": base_auc - tab_auc,
    **gate_faithfulness(ag_model, validation_ds, strength=1.0),
    **brier_ece(ag_model, validation_ds),
}

# Save diagnostics in run folder
with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
    json.dump(diag, f, indent=2)

display(pd.DataFrame([diag]))
